In [1]:
%pip install -q syncode==0.4.16 transformers==4.53.2 accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.1/206.1 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0,

In [2]:
import json
import operator
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from syncode import Grammar, SyncodeLogitsProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2")
model = AutoModelForCausalLM.from_pretrained("microsoft/phi-2", torch_dtype=dtype, low_cpu_mem_usage=True).to(device).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
grammar = r"""
start: "[" NUMBER OP NUMBER "]"
OP: "+" | "-" | "*" | "/"
NUMBER: /[0-9]+(\.[0-9]*)?/
%import common.WS
%ignore WS
"""
decoder = SyncodeLogitsProcessor(Grammar(grammar), tokenizer, parse_output_only=True, mode="grammar_strict")

prompt_prefix = """Extract the two operands and the operator from each question as [first operand operator second operand]. Do not compute the result.

Question: What is 8.5 plus 2.0?
Extraction: [8.5 + 2.0]

Question: Subtract 3.0 from 12.5.
Extraction: [12.5 - 3.0]

Question: What is 6.0 multiplied by 4.5?
Extraction: [6.0 * 4.5]

Question: What is 15.0 divided by 2.5?
Extraction: [15.0 / 2.5]

"""

[2026-10-26 02:47:05,131-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_6956270039_50257.pkl exist: False
[2026-10-26 02:47:05,137-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_6956270039_50257.pkl.
[2026-10-26 02:47:06,788-syncode.mask_store.fsm_set] - 5 FSMs with 17 states initialized in 0.00 seconds
[2026-10-26 02:47:06,789-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-26 02:47:06,790-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 25 to 9


100%|██████████| 17/17 [00:10<00:00,  1.70it/s]

[2026-10-26 02:47:16,823-syncode.mask_store.mask_store] - Time taken to create mask store: 11.64 seconds


In [10]:
import re

operations = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
extraction_re = re.compile(r"\s*\[\s*([0-9]+\.?[0-9]*)\s*([-+*/])\s*([0-9]+\.?[0-9]*)\s*\]")

def run_syncode(textual_query):
    prompt = prompt_prefix + f"Question: {textual_query}\nExtraction:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    decoder.reset()
    output = model.generate(**inputs, logits_processor=[decoder], max_new_tokens=40, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id, stop_strings=["]"], tokenizer=tokenizer)
    text = tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    a, op, b = extraction_re.match(text).groups()
    operands = [float(a), float(b)]
    return operands, op, float(operations[op](*operands))

In [11]:
queries = [
    "What is 327. multiplied by 11.0?",
    "What is 45.1 plus 23.54?",
    "What is 120.4 divided by 4.0?",
    "What is 9.5 minus 2.0?",
]
for query in queries:
    print(query)
    print(run_syncode(query))

What is 327. multiplied by 11.0?
([327.0, 11.0], '*', 3597.0)
What is 45.1 plus 23.54?
([45.1, 23.54], '+', 68.64)
What is 120.4 divided by 4.0?
([120.4, 4.0], '/', 30.1)
What is 9.5 minus 2.0?
([9.5, 2.0], '-', 7.5)
